# PySceneDetect

## Documentación

https://www.scenedetect.com/docs/latest/api

## Importar librerías

In [1]:
import numpy as np
from enum import Enum
from scenedetect import StatsManager,ContentDetector, AdaptiveDetector, SceneManager,detect, split_video_ffmpeg, open_video
from scenedetect.platform import init_logger
from scenedetect.scene_manager import get_scenes_from_cuts, Interpolation, save_images
from scenedetect.backends import AVAILABLE_BACKENDS

ModuleNotFoundError: No module named 'scenedetect'

In [3]:
# Initializes logging for PySceneDetect. The logger instance used is named ‘pyscenedetect’. By default the logger has no handlers to suppress output. All existing log handlers are replaced every time this function is invoked.
init_logger(log_level=20, show_stdout=False, log_file=None)

In [4]:
# All backends available on the current system can be found via AVAILABLE_BACKENDS.
AVAILABLE_BACKENDS

{'opencv': scenedetect.backends.opencv.VideoStreamCv2}

In [5]:
video_path = '../../_Videos_De_Clase/Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c].mp4'

In [6]:
OUTPUT_DIR = os.environ['OUTPUT_DIR']

In [7]:
STATS_FILE_PATH = os.environ['STATS_FILE_PATH']

In [8]:
video = open_video(video_path)

In [9]:
# Callback to invoke on the first frame of every new scene detection.
def on_new_scene(frame_img: np.ndarray, frame_num: int):
    print("New scene found at frame %d." % frame_num)

In [10]:
def print_scenes(scene_list: list):
    for i, (start_time, end_time) in enumerate(scene_list, start=1):
        print("Escena {}: {} - {}".format(i, start_time.get_timecode(), end_time.get_timecode()))

In [11]:
class Interpolation(Enum):
    INTERPOLATION_NEAREST = 0
    INTERPOLATION_LINEAR = 1
    INTERPOLATION_CUBIC = 2
    INTERPOLATION_AREA = 3
    INTERPOLATION_LANCZOS4 = 4

#### Resultados por tipo de interpolación (AdaptiveDetector)

Nearest: 0
Linear: n/a
Cubic: n/a
Area: n/a
Lanczos4: n/a

#### Resultados por tipo de interpolación (ContentDetector)

Nearest: 18 (27033)
Linear: 18 (27033)
Cubic: 18 (27033)
Area: 18 (27033)
Lanczos4: 18 (27033)

In [12]:
interpolation = Interpolation.INTERPOLATION_LANCZOS4.value

In [13]:
Interpolation(interpolation)

<Interpolation.INTERPOLATION_LANCZOS4: 4>

In [14]:
scene_manager = SceneManager(stats_manager=StatsManager())
scene_manager.add_detector(AdaptiveDetector(adaptive_threshold=2.0, min_scene_len=10))

In [15]:
# Detect all scenes in video from current position to end.
scene_manager.detect_scenes(video, callback=on_new_scene, show_progress=True)

  Detected: 1 | Progress:  18%|█▊        | 2924/16610 [00:03<00:15, 900.45frames/s]New scene found at frame 2759.
  Detected: 2 | Progress:  33%|███▎      | 5417/16610 [00:05<00:12, 925.99frames/s]New scene found at frame 5303.
  Detected: 3 | Progress:  35%|███▍      | 5794/16610 [00:06<00:11, 931.47frames/s]New scene found at frame 5667.
  Detected: 5 | Progress:  48%|████▊     | 7965/16610 [00:08<00:09, 906.27frames/s]New scene found at frame 7833.
New scene found at frame 7895.
  Detected: 6 | Progress:  49%|████▉     | 8153/16610 [00:08<00:09, 916.20frames/s]New scene found at frame 8042.
  Detected: 7 | Progress:  53%|█████▎    | 8809/16610 [00:09<00:08, 906.01frames/s]New scene found at frame 8630.
  Detected: 8 | Progress:  56%|█████▋    | 9370/16610 [00:10<00:07, 935.07frames/s]New scene found at frame 9226.
  Detected: 10 | Progress:  58%|█████▊    | 9662/16610 [00:10<00:07, 953.21frames/s]New scene found at frame 9515.
New scene found at frame 9541.
  Detected: 12 | Progress

16610

In [16]:
scene_list = scene_manager.get_scene_list()
print(scene_list)
print_scenes(scene_list=scene_list)

[(00:00:00.000 [frame=0, fps=30.000], 00:01:31.967 [frame=2759, fps=30.000]), (00:01:31.967 [frame=2759, fps=30.000], 00:02:56.767 [frame=5303, fps=30.000]), (00:02:56.767 [frame=5303, fps=30.000], 00:03:08.900 [frame=5667, fps=30.000]), (00:03:08.900 [frame=5667, fps=30.000], 00:04:21.100 [frame=7833, fps=30.000]), (00:04:21.100 [frame=7833, fps=30.000], 00:04:23.167 [frame=7895, fps=30.000]), (00:04:23.167 [frame=7895, fps=30.000], 00:04:28.067 [frame=8042, fps=30.000]), (00:04:28.067 [frame=8042, fps=30.000], 00:04:47.667 [frame=8630, fps=30.000]), (00:04:47.667 [frame=8630, fps=30.000], 00:05:07.533 [frame=9226, fps=30.000]), (00:05:07.533 [frame=9226, fps=30.000], 00:05:17.167 [frame=9515, fps=30.000]), (00:05:17.167 [frame=9515, fps=30.000], 00:05:18.033 [frame=9541, fps=30.000]), (00:05:18.033 [frame=9541, fps=30.000], 00:06:11.267 [frame=11138, fps=30.000]), (00:06:11.267 [frame=11138, fps=30.000], 00:06:13.233 [frame=11197, fps=30.000]), (00:06:13.233 [frame=11197, fps=30.000]

In [17]:
# Save per-frame statistics to disk.
scene_manager.stats_manager.save_to_csv(csv_file=STATS_FILE_PATH)

In [18]:
# Save a set number of images from each scene, given a list of scenes and the associated video/frame source.
save_images(video=video, scene_list=scene_list, output_dir=OUTPUT_DIR, image_extension='webp', encoder_param=80, show_progress=True)

100%|██████████| 60/60 [00:13<00:00,  4.60images/s]


{0: ['Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-001-01.webp',
  'Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-001-02.webp',
  'Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-001-03.webp'],
 1: ['Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-002-01.webp',
  'Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-002-02.webp',
  'Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-002-03.webp'],
 2: ['Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0wjGB-c]-Scene-003-01.webp',
  'Introducción a la Algoritmia - Cápsula Técnica 20 (C20) - Comparar, unir y copiar cadenas [wF6B0w

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8e809eb8-650a-4468-a096-2a2830a4a1e1' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>